In [12]:
import os
import PIL
device='cuda'
from PIL import Image
from torch.utils.data import Dataset,DataLoader
from torch.nn import Module
from torch.optim import AdamW
import json
import torch
import torch.nn.functional as F
from tqdm import tqdm
import matplotlib.pyplot as plt
from torch import nn
from torchvision import transforms

In [13]:
alphabet=[symb for symb in '_ABEKMHOPCTYX0123456789']
let2int={i:let for let,i in enumerate(alphabet)}
int2let={let:i for let,i in enumerate(alphabet)}

In [14]:


class NumberDataset(Dataset):
    def __init__(self,path,number_len,let2int):
        super(NumberDataset,self).__init__()
        self.number_len=number_len
        img_path=os.path.join(path,'img')
        label_path=os.path.join(path,'ann')
        self.let2int=let2int

        self.images=[os.path.join(img_path,img) for img in os.listdir(img_path)]
        self.labels=[os.path.join(label_path,label) for label in os.listdir(label_path)]
        
        self.images.sort(reverse=True)
        self.labels.sort(reverse=True)

        self.trans=transforms.Compose([
            transforms.Resize((64,128)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.images)
    def __getitem__(self,idx):
        idx_img=Image.open(self.images[idx]).convert('RGB')
        idx_label=self.labels[idx]
        
        with open(idx_label,'r') as file_option:
            jf=json.load(file_option)
            #return jf['name'][0:self.number_len]
            tensor_label=torch.tensor([self.let2int[let] for let in jf['description'][0:self.number_len] if let!='_'])
        tensor_img=self.trans(idx_img)
        return {
            'img':tensor_img,
            'label':tensor_label,
            'label_len':len(tensor_label)
        }
    


In [15]:
def collate_fn(batch):
    imgs = torch.stack([x['img'] for x in batch])
    labels=[x['label'] for x in batch]
    label_lens=torch.tensor([x['label_len'] for x in batch])
    label = torch.cat(labels)
    return imgs,label,label_lens

In [17]:
number_data=NumberDataset(path='//home/artemybombastic/MyGit/VehicleNumberData/VNR_Data/train',number_len=9,let2int=let2int)
number_dataloader=DataLoader(number_data,batch_size=16,shuffle=False,drop_last=True,collate_fn=collate_fn)


In [18]:
class ResnetBlock(Module):
    def __init__(self,input_size,output_size,stride=1,downsample=None):#downsample нужно в случае понижения размерности блока):
        super().__init__()
        self.act=nn.ReLU(inplace=True)
        self.conv0=nn.Conv2d(input_size,output_size,kernel_size=3,stride=stride,padding=1)
        self.norm0=nn.BatchNorm2d(output_size)

        self.conv1=nn.Conv2d(output_size,output_size,kernel_size=3,stride=1,padding=1)
        self.norm1=nn.BatchNorm2d(output_size)

        self.downsample=downsample
    def forward(self,x):
        out=self.conv0(x)
        out=self.norm0(out)
        out=self.act(out)
        out=self.conv1(out)
        out=self.norm1(out)
        if self.downsample:
            x=self.downsample(x)
        out+=x
        out=self.act(out)

        return out

In [19]:
def make_layers(block,cnt,input_size,output_size,stride=1,downsample=False):
    blocks=[]

    if downsample or input_size!=output_size or stride!=1:
        downsample=nn.Sequential(
            nn.Conv2d(input_size,output_size,1,stride,bias=False),
            nn.BatchNorm2d(output_size)
        )

    blocks.append(block(input_size,output_size,stride,downsample))    
    for i in range(1,cnt):
        blocks.append(block(output_size,output_size))

    return nn.Sequential(*blocks)
    

In [20]:
class Resnet34(Module):
    def __init__(self,input_size,hidden_size):
        super().__init__()

        self.initial_lay=nn.Sequential(
            nn.Conv2d(input_size,hidden_size,kernel_size=7,stride=2,padding=3),
            nn.BatchNorm2d(hidden_size),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3,stride=2,padding=1)
        )

        self.lay0=make_layers(block=ResnetBlock,cnt=3,input_size=hidden_size,output_size=hidden_size,downsample=False)
        self.lay1=make_layers(block=ResnetBlock,cnt=4,input_size=hidden_size,output_size=hidden_size*2,stride=(2,1),downsample=True)
        self.lay2=make_layers(block=ResnetBlock,cnt=6,input_size=hidden_size*2,output_size=hidden_size*4,stride=(2,1),downsample=True)
        self.lay3=make_layers(block=ResnetBlock,cnt=3,input_size=hidden_size*4,output_size=hidden_size*8,stride=(2,1),downsample=True)

        self.avg_pool=nn.AdaptiveAvgPool2d((1,None))
        
    def forward(self,x):
        #print(x.shape)
        out=self.initial_lay(x)
        #print(out.shape)
        #print('СЛОИ:')
        out=self.lay0(out)
        #print(out.shape)
        out=self.lay1(out)
        #print(out.shape)
        out=self.lay2(out)
        #print(out.shape)
        out=self.lay3(out)
        #print(out.shape)

        final_out=self.avg_pool(out)
        #print(final_out.shape)
        return final_out

In [21]:
class CRNN(Module):
    def __init__(self,input_size,hidden_size,out_size):
        super().__init__()

        self.stn=STN()

        self.cnn=Resnet34(input_size,hidden_size)
        self.rnn=nn.LSTM(hidden_size*8,hidden_size*4,num_layers=1,bidirectional=True)
        self.final_lay=nn.Sequential(    
            nn.Dropout(p=0.2),
            nn.Linear(hidden_size*8,out_size)
        )
                
    def forward(self,x):
        x=self.stn(x)
        out=self.cnn(x)

        out=out.squeeze(2).permute(2,0,1)
        out,_=self.rnn(out)

        out=self.final_lay(out)
        return out

In [22]:
class STN(Module):
    def __init__(self):
        super().__init__()

        self.localization= nn.Sequential(
            nn.Conv2d(3,8,kernel_size=7),
            nn.MaxPool2d(2,stride=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(8,10,kernel_size=5),
            nn.MaxPool2d(2,stride=2),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((3,3))
        )

        

        self.fc_loc=nn.Sequential(
            nn.Linear(10*3*3,32),
            nn.ReLU(inplace=True),
            nn.Linear(32,3*2)
        )
        #print(self.fc_loc)
        self.fc_loc[2].weight.data.zero_()
        self.fc_loc[2].bias.data.copy_(torch.tensor([1, 0, 0, 0, 1, 0], dtype=torch.float))
        #print(self.fc_loc)
    def forward(self,x):
        xs=self.localization(x)
        xs=xs.view(-1,10*3*3)
        theta=self.fc_loc(xs)
        theta=theta.view(-1,2,3)

        grid=F.affine_grid(theta,x.size())
        x=F.grid_sample(x,grid)
        return x
        

In [23]:
loss_fn=nn.CTCLoss(blank=0)
model=CRNN(input_size=3,hidden_size=64,out_size=len(alphabet)).to(device)

#weight_path='/home/artemybombastic/MyGit/VehicleNumberData/VNR_Data/weights/crnn_weights'

if f'crnn_weights.pth' in os.listdir('../VehicleNumberData/VNR_Data/weights/'):
    weights_dict=torch.load(f'../VehicleNumberData/VNR_Data/weights/crnn_weights.pth',weights_only=True)
    model.load_state_dict(weights_dict)
    print('Веса обнаружены')
optimizer=AdamW(model.parameters())

Веса обнаружены


In [29]:
import torch
from tqdm import tqdm


model.train()

dataloader=number_dataloader

losses=[]
for batch in (pbar:=tqdm(dataloader)):
    optimizer.zero_grad()
    img,label,label_len=batch
    pred=model(img.to(device))

    T = pred.size(0)
    N = pred.size(1)
    input_len = torch.full(size=(N,), fill_value=T, dtype=torch.int32)
    
    pred=pred.log_softmax(dim=2)
    loss=loss_fn(pred,label,input_len,label_len)
    loss.backward()
    loss_item=loss.item()
    losses.append(loss_item)
    optimizer.step()
    pbar.set_description(f"loss: {loss_item}")
try:
    torch.save(model.state_dict(),r'../VehicleNumberData/VNR_Data/weights/crnn_weights.pth')
except:
    print('Ошибка загрузки')
print(f'mean_loss: {sum(losses)/len(losses)}')


  0%|          | 0/2776 [00:00<?, ?it/s]/usr/lib/python3.14/site-packages/torch/nn/functional.py:5167: UserWarning: Default grid_sample and affine_grid behavior has changed to align_corners=False since 1.3.0. Please specify align_corners=True if the old behavior is desired. See the documentation of grid_sample for details.
  warnings.warn(
/usr/lib/python3.14/site-packages/torch/nn/functional.py:5100: UserWarning: Default grid_sample and affine_grid behavior has changed to align_corners=False since 1.3.0. Please specify align_corners=True if the old behavior is desired. See the documentation of grid_sample for details.
  warnings.warn(
loss: 0.0003202195221092552:   3%|▎         | 74/2776 [00:04<02:51, 15.72it/s] 


KeyboardInterrupt: 

In [ ]:
def ctc_decoder(pred_string):
    new_string=[]
    perv_symb=-1
    for symb in pred_string:
        if symb.item()!=perv_symb:
            if symb.item()!=0:
                new_string.append(int2let[symb.item()])
        perv_symb=symb
    return ''.join(new_string)

In [146]:

model.eval()

all_accuracy=[]
for batch in (pbar:=tqdm(dataloader)):
    img,label,label_len=batch
    pred=model(img.to(device))

    #форматирование label
    label=[num.item() for num in label]
    corected_label=[]
    for lenght in label_len:
        new_label=label[:lenght]
        new_label=[int2let[num] for num in new_label]
        new_label=''.join(new_label)
        corected_label.append(new_label)
        label=label[lenght:]

    #форматирование pred
    corected_pred=[ctc_decoder(word) for word in pred.argmax(dim=2).permute(1,0)]
    accuracy=[corected_label[i]==corected_pred[i] for i in range(16)]
    accuracy=sum(accuracy)/len(accuracy)
    
    all_accuracy.append(accuracy)
    pbar.set_description(f"accuracy: {accuracy}")
print(f'Средняя точность на тестовой выборке равна {sum(all_accuracy)/len(all_accuracy)}')

accuracy: 0.8125: 100%|██████████| 2776/2776 [01:48<00:00, 25.60it/s]

Средняя точность на тестовой выборке равна 0.9888778818443804
